# EfficientNet-B0 External Dataset (35 Vehicle Makes): Diagnostics, Interpretability & Feature Analysis
---
This notebook delivers a comprehensive diagnostic evaluation, explainability audit, and latent space analysis of the fine-tuned **EfficientNet-B0** model on the **External Dataset 35-make vehicle classification benchmark** (input resolution: **640×640**).

All core computational logic, feature extractors, and plotting pipelines are modularized in [`output_analysis_external.py`](output_analysis_external.py) / [`output_analysis.py`](../output_analysis.py).

### Analysis Modules:
1. **Model Architecture & Checkpoint Audit**: Verify layer topology, parameter counts, and target feature map layers.
2. **Global & Per-Class Performance Metrics**: Comprehensive test metrics (Accuracy, Top-5 Accuracy, Macro/Weighted F1, Loss) and interactive classification report.
3. **Confusion Matrix & Misclassification Patterns**: 35×35 normalized confusion heatmap, top error pairs, and platform-sharing analysis.
4. **Hardest Misclassifications (High-Confidence Errors)**: Visual inspection of cases where the model was strongly confident but incorrect.
5. **Grad-CAM Explainability (Highest-Confidence True vs. False)**:
   - Visualizing model attention on correct predictions (iconic grilles, badges, headlights).
   - Comparative Grad-CAM on hardest false predictions (misleading visual cues vs. evidence for ground truth).
6. **Latent Feature Space Visualisation (t-SNE & PCA)**:
   - Extracting 1,280-dimensional penultimate embeddings from `GlobalAveragePooling2D`.
   - 2D manifold projection demonstrating cluster separation and inter-class geometric overlap.
7. **Prediction Confidence & Calibration Analysis**:
   - Softmax probability distributions for correct vs incorrect predictions and confidence threshold rejection curves.
8. **Executive Summary & Actionable Recommendations**.


In [ ]:
import sys
from pathlib import Path

# Ensure notebook directory is in python search path
CURRENT_DIR = Path("/home/researchadmin/Econ/repo-clone/stanford-cars-model/code/current")
if str(CURRENT_DIR) not in sys.path:
    sys.path.insert(0, str(CURRENT_DIR))
if str(CURRENT_DIR / "external_dataset") not in sys.path:
    sys.path.insert(0, str(CURRENT_DIR / "external_dataset"))

# Import the External Dataset diagnostic engine adapter
import output_analysis_external as oam

# Configure environment, plotting styles, and hardware
oam.setup_environment()


## 1. Model Loading & Architecture Inspection
---
We load the best checkpoint saved during training (`val_accuracy` monitoring).
Let us inspect:
- Input resolution: **640×640×3**
- Feature map dimension from the final convolutional layer (`top_activation`): **20×20×1280**
- Latent feature representation (`avg_pool`): **1280-D vector**
- Softmax prediction head: **35 classes**

In [ ]:
model, model_info = oam.load_analysis_model()

## 2. Test Set Evaluation & Metrics Breakdown
---
We evaluate performance on the held-out **Test split (7,815 images)** across all 35 makes.
We load the evaluation cache and test reports generated during model evaluation.

In [ ]:
test_df, y_true, y_pred, test_metrics = oam.load_test_evaluation(model=model)

In [ ]:
class_report, summary_table = oam.load_classification_report_df()

## 3. Confusion Matrix & Misclassification Patterns
---
Let us analyze the normalized confusion matrix across all 35 makes to detect systematic patterns:
- **Shared Automotive Platforms**: Stellantis siblings (Dodge ↔ Chrysler), General Motors (Buick ↔ Chevrolet), Honda ↔ Acura, Toyota ↔ Lexus, Hyundai ↔ Kia.
- **Segment Overlap**: Full-size pickup trucks and SUVs across domestic manufacturers.

In [ ]:
cm_norm, top_errors_df = oam.plot_confusion_matrix_and_top_errors(y_true, y_pred, top_k=15)

## 4. Hardest Misclassifications (High-Confidence Errors)
---
A critical failure mode in computer vision is when the model is **highly confident but wrong** ($P(\hat{y} \mid x) \gg 0.5$, $\hat{y} \ne y$).
We evaluate a balanced sample across makes, identify high-confidence false predictions, and inspect the specific vehicles to diagnose why the model was misled.

In [ ]:
sample_cache = oam.get_or_compute_sample_cache(model, test_df)

## 5. Grad-CAM Explainability (Highest-Confidence True vs. Highest-Confidence False)
---
Grad-CAM (**Gradient-Weighted Class Activation Mapping**) calculates the gradients of any target class score $y^c$ with respect to the final convolutional feature maps $A^k$ (`top_activation`):
$$\alpha_k^c = \frac{1}{Z} \sum_{i} \sum_{j} \frac{\partial y^c}{\partial A^k_{i,j}}, \quad L^c_{\text{Grad-CAM}} = \text{ReLU}\left( \sum_k \alpha_k^c A^k \right)$$

Because Grad-CAM is **class-specific**, we can examine:
1. **Highest-Confidence TRUE Predictions**: Where does the network look when it achieves **100% certainty** on the correct make?
2. **Highest-Confidence FALSE Predictions (Hardest Errors)**: Which misleading visual cues fooled the model into high false certainty ($\ge 80\%$ to $99\%$ false confidence), and what evidence did it find for the ground-truth class?

In [ ]:
grad_analyzer = oam.GradCAMAnalyzer(model)

In [ ]:
# 5.1 Grad-CAM on Highest-Confidence TRUE Predictions (100.0% Confidence)
grad_analyzer.plot_highest_confidence_true(sample_cache)

In [ ]:
# 5.2 Comparative Grad-CAM on Highest-Confidence FALSE Predictions (Hardest Errors)
grad_analyzer.plot_comparative_highest_confidence_false(sample_cache, n_samples=4)

### 5.3 Interactive Grad-CAM Query by Vehicle Make and Model
---
You can generate Grad-CAM visual explainability for **any vehicle make and model** in the dataset:
- To inspect all available models for a brand: `oam.list_available_models(test_df, "Porsche")`
- To run Grad-CAM on specific models: `grad_analyzer.plot_gradcam_by_make_model(test_df, make="Porsche", model_name="Cayenne", n_samples=2)`
- Supports case-insensitive and partial model matching (e.g. `cayenne`, `wrangler`, `d15`, `civic`).
- If the model correctly classified the vehicle, a clean 2-column view (Original vs. Grad-CAM) is rendered. If misclassified, it automatically displays the 3-panel comparative diagnostic view.

In [ ]:
# Inspect available models for target makes
oam.list_available_models(test_df, "Porsche")
oam.list_available_models(test_df, "Jeep")

In [ ]:
# Generate Grad-CAM for specific make & model pairs
# Example 1: BMW
grad_analyzer.plot_gradcam_by_make_model(test_df, make="BMW", model_name="X1", n_samples=2)

# Example 2: Jeep Wrangler
grad_analyzer.plot_gradcam_by_make_model(test_df, make="Jeep", model_name="Wrangler", n_samples=2)

# Example 3: Porsche
grad_analyzer.plot_gradcam_by_make_model(test_df, make="Porsche", n_samples=2)

# Example 4: Ford
grad_analyzer.plot_gradcam_by_make_model(test_df, make="Ford", n_samples=2)

## 6. Latent Feature Space Visualisation (t-SNE & PCA)
---
We construct a **Feature Extractor** that maps vehicle images to their **1,280-dimensional embedding vector** (output of the `avg_pool` layer).
We analyze the geometric structure of this high-dimensional latent space using:
- **t-SNE (t-Distributed Stochastic Neighbor Embedding)**: Preserves local neighborhood topology, revealing discrete clusters of makes with distinct styling.
- **PCA (Principal Component Analysis)**: Preserves global variance axes across all vehicle makes.

In [ ]:
oam.plot_latent_space_tsne_pca(sample_cache)

## 7. Prediction Confidence & Calibration Analysis
---
We examine the distribution of softmax confidence scores between **correct predictions** and **misclassifications** to evaluate model calibration and the effectiveness of confidence-based rejection thresholds.

In [ ]:
oam.plot_confidence_calibration(sample_cache)

## 8. Executive Summary & Actionable Takeaways
---

### Key Strengths:
1. **Outstanding Benchmark Accuracy**: Overall accuracy of **94.18%**, Top-5 Accuracy of **98.39%**, and Weighted F1 of **94.19%** across all 35 makes at 640×640 input resolution on 7,815 test images.
2. **Robust Multi-Source Fusion**: Trained seamlessly across BoxCars116k, Stanford-Cars, and CompCars (Surveillance CCTV + Web) with peak-frame selection.
3. **Interpretable Attention (Grad-CAM)**: The model consistently attends to front grilles, emblem badges, headlamp signatures, and window trim lines.
4. **Well-Calibrated Confidence**: Correct predictions average high confidence (>95%), whereas misclassifications exhibit lower confidence, enabling reliable rejection thresholds.

### Actionable Next Steps:
- **Production Deployment**: Use the optimized ONNX model (`efficientnet_b0_best.onnx`) with `infer_merged.py` for ultra-low latency inference (~10-20 ms/image).
- **Thresholding Strategy**: Set a prediction confidence cutoff at $\ge 0.70$ to flag ambiguous or cropped silhouettes for human review.